In [78]:
import pandas as pd
df = pd.read_csv('mtsamples.csv', index_col=0)

In [79]:
df.shape
df.head()
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4999 entries, 0 to 4998
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   description        4999 non-null   object
 1   medical_specialty  4999 non-null   object
 2   sample_name        4999 non-null   object
 3   transcription      4966 non-null   object
 4   keywords           3931 non-null   object
dtypes: object(5)
memory usage: 234.3+ KB


In [80]:
counts=df['medical_specialty'].value_counts()
counts

,count
medical_specialty,
Surgery,1103
Consult - History and Phy.,516
Cardiovascular / Pulmonary,372
Orthopedic,355
Radiology,273
General Medicine,259
Gastroenterology,230
Neurology,223
SOAP / Chart / Progress Notes,166


In [81]:
df['medical_specialty'].nunique()

40

In [82]:
base = counts.iloc[0]

ratio = counts / base
print(ratio)

medical_specialty
Surgery                          1.000000
Consult - History and Phy.       0.467815
Cardiovascular / Pulmonary       0.337262
Orthopedic                       0.321850
Radiology                        0.247507
General Medicine                 0.234814
Gastroenterology                 0.208522
Neurology                        0.202176
SOAP / Chart / Progress Notes    0.150499
Obstetrics / Gynecology          0.145059
Urology                          0.143246
Discharge Summary                0.097915
ENT - Otolaryngology             0.088849
Neurosurgery                     0.085222
Hematology - Oncology            0.081596
Ophthalmology                    0.075249
Nephrology                       0.073436
Emergency Room Reports           0.067996
Pediatrics - Neonatal            0.063463
Pain Management                  0.056210
Psychiatry / Psychology          0.048051
Office Notes                     0.046238
Podiatry                         0.042611
Dermatology     

In [83]:
df['medical_specialty'] = df['medical_specialty'].str.strip()
top_classes = df['medical_specialty'].value_counts().index[:19]
dff = df[df['medical_specialty'].isin(top_classes)]
dff_counts=dff['medical_specialty'].value_counts()
dff_counts

,count
medical_specialty,
Surgery,1103
Consult - History and Phy.,516
Cardiovascular / Pulmonary,372
Orthopedic,355
Radiology,273
General Medicine,259
Gastroenterology,230
Neurology,223
SOAP / Chart / Progress Notes,166


In [84]:
dff.shape

(4514, 5)

In [85]:
dff=dff.dropna(subset=['transcription'])
dff.shape

(4483, 5)

In [86]:
dff['sample_name'].head(10)

,sample_name
3,2-D Echocardiogram - 1
4,2-D Echocardiogram - 2
7,2-D Echocardiogram - 3
9,2-D Echocardiogram - 4
11,2-D Doppler
12,Moyamoya Disease
16,Tracheostomy
18,Vasectomy - 4
19,Airway Compromise & Foreign Body - ER Visit
20,Whole Body Radionuclide Bone Scan


In [87]:
dff = dff[['transcription', 'medical_specialty']]
dff.shape

(4483, 2)

In [88]:
print(dff['transcription'].iloc[0])

2-D M-MODE: , ,1.  Left atrial enlargement with left atrial diameter of 4.7 cm.,2.  Normal size right and left ventricle.,3.  Normal LV systolic function with left ventricular ejection fraction of 51%.,4.  Normal LV diastolic function.,5.  No pericardial effusion.,6.  Normal morphology of aortic valve, mitral valve, tricuspid valve, and pulmonary valve.,7.  PA systolic pressure is 36 mmHg.,DOPPLER: , ,1.  Mild mitral and tricuspid regurgitation.,2.  Trace aortic and pulmonary regurgitation.


In [89]:
import re
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('omw-1.4', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [90]:
def get_wordnet_pos(tag):

    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

MEDICAL_HEADERS = re.compile(
    r'\b(subjective|objective|assessment|plan|history|diagnosis|procedure|description)\s*:',
    re.IGNORECASE
)


def clean_text(text):
    text = str(text).lower()
    text = MEDICAL_HEADERS.sub(' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word not in stop_words and (len(word) > 2 or (word.isdigit() and len(word) >= 2))
    ]
    return ' '.join(tokens)

In [91]:
print(clean_text(dff['transcription'].iloc[0]))

mode leave atrial enlargement left atrial diameter normal size right leave ventricle normal systolic function left ventricular ejection fraction 51 normal diastolic function pericardial effusion normal morphology aortic valve mitral valve tricuspid valve pulmonary valve systolic pressure 36 mmhg doppler mild mitral tricuspid regurgitation trace aortic pulmonary regurgitation


In [92]:
dff['clean_transcription']=dff['transcription'].apply(clean_text)
dff.head(5)

,transcription,medical_specialty,clean_transcription
3,"2-D M-MODE: , ,1. Left atrial enlargement wit...",Cardiovascular / Pulmonary,mode leave atrial enlargement left atrial diam...
4,1. The left ventricular cavity size and wall ...,Cardiovascular / Pulmonary,left ventricular cavity size wall thickness ap...
7,"2-D ECHOCARDIOGRAM,Multiple views of the heart...",Cardiovascular / Pulmonary,echocardiogram multiple view heart great vesse...
9,"DESCRIPTION:,1. Normal cardiac chambers size....",Cardiovascular / Pulmonary,normal cardiac chamber size normal leave ventr...
11,"2-D STUDY,1. Mild aortic stenosis, widely calc...",Cardiovascular / Pulmonary,study mild aortic stenosis widely calcify mini...


In [93]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2), min_df=5)
X = vectorizer.fit_transform(dff['clean_transcription'])
dff['medical_specialty'] = dff['medical_specialty'].str.strip()
y=dff['medical_specialty']

In [94]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(3586, 1000) (3586,)
(897, 1000) (897,)


In [122]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=100, class_weight='balanced', C=0.1, penalty='l2', solver='liblinear')
model.fit(X_train, y_train)

LogisticRegression(C=0.1, class_weight='balanced', solver='liblinear')

In [123]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
param_grid = [
    {'solver': ['lbfgs', 'saga'], 'penalty': ['l2'], 'C': [0.001, 0.01, 0.1, 1.0], 'max_iter': [100, 500, 1000]},
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.001, 0.01, 0.1, 1.0], 'max_iter': [100, 500, 1000]},
]
lr_base = LogisticRegression(max_iter=1000, class_weight='balanced')
grid_search = GridSearchCV(lr_base, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-validation score: 0.4110375558948419


In [124]:
best_model = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_model.predict(X_test))
train_acc = accuracy_score(y_train, best_model.predict(X_train))
print("Train:", train_acc)
print("Test:", test_acc)
print("Gap:", train_acc - test_acc)

Train: 0.4712771890686001
Test: 0.41471571906354515
Gap: 0.05656147000505496


In [125]:
from sklearn.metrics import accuracy_score, classification_report
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.41471571906354515
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.40      0.39      0.39        74
   Consult - History and Phy.       0.43      0.51      0.47       103
            Discharge Summary       0.47      0.91      0.62        22
         ENT - Otolaryngology       0.57      0.63      0.60        19
       Emergency Room Reports       0.23      0.53      0.32        15
             Gastroenterology       0.45      0.40      0.42        45
             General Medicine       0.27      0.13      0.18        52
        Hematology - Oncology       0.29      0.22      0.25        18
                   Nephrology       0.31      0.31      0.31        16
                    Neurology       0.40      0.42      0.41        45
                 Neurosurgery       0.30      0.68      0.42        19
      Obstetrics / Gynecology       0.52      0.55      0.53        31
                Ophthalmology       0.52      0.94      

In [126]:
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)
print("Train accuracy:", accuracy_score(y_train, train_preds))
print("Test accuracy:", accuracy_score(y_test, test_preds))

Train accuracy: 0.4712771890686001
Test accuracy: 0.41471571906354515


In [127]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted')
print(scores)
print(scores.mean())

[0.42682207 0.39471973 0.40422158 0.4113229  0.38797647]
0.40501255057065466


In [128]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [129]:
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.40022296544035674
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.44      0.42      0.43        74
   Consult - History and Phy.       0.31      0.88      0.46       103
            Discharge Summary       0.38      0.27      0.32        22
         ENT - Otolaryngology       0.00      0.00      0.00        19
       Emergency Room Reports       0.00      0.00      0.00        15
             Gastroenterology       0.48      0.24      0.32        45
             General Medicine       0.00      0.00      0.00        52
        Hematology - Oncology       0.00      0.00      0.00        18
                   Nephrology       0.00      0.00      0.00        16
                    Neurology       0.41      0.38      0.40        45
                 Neurosurgery       0.00      0.00      0.00        19
      Obstetrics / Gynecology       0.60      0.10      0.17        31
                Ophthalmology       0.00      0.00      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [130]:
from sklearn.svm import LinearSVC
model = LinearSVC(class_weight='balanced', C=0.01, loss='squared_hinge', max_iter=500)
model.fit(X_train, y_train)

LinearSVC(C=0.01, class_weight='balanced', max_iter=500)

In [131]:
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.40468227424749165
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.40      0.46      0.42        74
   Consult - History and Phy.       0.38      0.29      0.33       103
            Discharge Summary       0.43      0.91      0.59        22
         ENT - Otolaryngology       0.59      0.84      0.70        19
       Emergency Room Reports       0.19      0.53      0.28        15
             Gastroenterology       0.43      0.44      0.44        45
             General Medicine       0.26      0.13      0.18        52
        Hematology - Oncology       0.27      0.33      0.30        18
                   Nephrology       0.32      0.44      0.37        16
                    Neurology       0.41      0.51      0.46        45
                 Neurosurgery       0.29      0.74      0.41        19
      Obstetrics / Gynecology       0.47      0.61      0.54        31
                Ophthalmology       0.48      0.94      

In [121]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0],
    'loss': ['hinge', 'squared_hinge'],
    'max_iter': [100, 500, 1000]
}
svc_base = LinearSVC(class_weight='balanced')
grid_search_svc = GridSearchCV(svc_base, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search_svc.fit(X_train, y_train)
print("Best parameters:", grid_search_svc.best_params_)
print("Best CV score:", grid_search_svc.best_score_)

Best parameters: {'C': 0.001, 'loss': 'hinge', 'max_iter': 500}
Best CV score: 0.3998850052252693


In [105]:
for c in [0.01, 0.05, 0.1, 0.5, 1.0]:
    svm = LinearSVC(class_weight='balanced', C=c)
    svm.fit(X_train, y_train)
    preds = svm.predict(X_test)
    print(f"C={c}: {accuracy_score(y_test, preds):.4f}")

C=0.01: 0.4047
C=0.05: 0.3779
C=0.1: 0.3612
C=0.5: 0.2798
C=1.0: 0.2185


In [132]:
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)
print("Train accuracy:", accuracy_score(y_train, train_preds))
print("Test accuracy:", accuracy_score(y_test, test_preds))

Train accuracy: 0.4617958728388176
Test accuracy: 0.40468227424749165


In [107]:
for c in [0.01, 0.05, 0.1, 0.5, 1.0]:
    model=LogisticRegression(max_iter=1000, class_weight='balanced', C=c)
    model.fit(X_train, y_train)
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    print(f"C={c} -> Train: {accuracy_score(y_train, train_preds):.4f}, Test: {accuracy_score(y_test, test_preds):.4f}, Gap: {accuracy_score(y_train, train_preds) - accuracy_score(y_test, test_preds):.4f}")

C=0.01 -> Train: 0.3996, Test: 0.3724, Gap: 0.0273
C=0.05 -> Train: 0.4163, Test: 0.3824, Gap: 0.0340
C=0.1 -> Train: 0.4283, Test: 0.3835, Gap: 0.0448
C=0.5 -> Train: 0.4643, Test: 0.3891, Gap: 0.0752
C=1.0 -> Train: 0.4810, Test: 0.3768, Gap: 0.1042


In [108]:
from sklearn.linear_model import PassiveAggressiveClassifier
model = PassiveAggressiveClassifier(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.1426978818283166
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.15      0.18      0.16        74
   Consult - History and Phy.       0.20      0.17      0.18       103
            Discharge Summary       0.23      0.27      0.25        22
         ENT - Otolaryngology       0.24      0.26      0.25        19
       Emergency Room Reports       0.06      0.07      0.06        15
             Gastroenterology       0.16      0.20      0.18        45
             General Medicine       0.06      0.06      0.06        52
        Hematology - Oncology       0.00      0.00      0.00        18
                   Nephrology       0.07      0.06      0.06        16
                    Neurology       0.22      0.29      0.25        45
                 Neurosurgery       0.00      0.00      0.00        19
      Obstetrics / Gynecology       0.06      0.06      0.06        31
                Ophthalmology       0.19      0.25      0

In [109]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

0.13377926421404682
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.10      0.09      0.10        74
   Consult - History and Phy.       0.26      0.30      0.28       103
            Discharge Summary       0.21      0.27      0.24        22
         ENT - Otolaryngology       0.21      0.16      0.18        19
       Emergency Room Reports       0.00      0.00      0.00        15
             Gastroenterology       0.11      0.11      0.11        45
             General Medicine       0.04      0.04      0.04        52
        Hematology - Oncology       0.00      0.00      0.00        18
                   Nephrology       0.00      0.00      0.00        16
                    Neurology       0.15      0.18      0.16        45
                 Neurosurgery       0.00      0.00      0.00        19
      Obstetrics / Gynecology       0.00      0.00      0.00        31
                Ophthalmology       0.17      0.19      

In [110]:
#label encoder for XG_Boost
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [111]:
from xgboost import XGBClassifier
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train_encoded)
predictions = model.predict(X_test)
print(accuracy_score(y_test_encoded, predictions))
print(classification_report(y_test_encoded, predictions))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:17:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.13377926421404682
              precision    recall  f1-score   support

           0       0.12      0.12      0.12        74
           1       0.22      0.27      0.24       103
           2       0.20      0.23      0.21        22
           3       0.23      0.16      0.19        19
           4       0.00      0.00      0.00        15
           5       0.12      0.11      0.12        45
           6       0.02      0.02      0.02        52
           7       0.00      0.00      0.00        18
           8       0.00      0.00      0.00        16
           9       0.16      0.13      0.14        45
          10       0.00      0.00      0.00        19
          11       0.00      0.00      0.00        31
          12       0.17      0.19      0.18        16
          13       0.06      0.06      0.06        71
          14       0.00      0.00      0.00        14
          15       0.16      0.15      0.15        55
          16       0.11      0.12      0.11        33
       

In [133]:
from sklearn.metrics import accuracy_score
import numpy as np

def confidence_interval(acc, n):
    margin = 1.96 * np.sqrt((acc * (1 - acc)) / n)
    return acc - margin, acc + margin

n = len(y_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1, penalty='l2', solver='liblinear'),
    'LinearSVC': LinearSVC(class_weight='balanced', C=0.01, loss='squared_hinge', max_iter=500),
    'Naive Bayes': MultinomialNB(),
    'Passive Aggressive': PassiveAggressiveClassifier(class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(class_weight='balanced')
}

print(f"{'Model':30s} | {'Train':6s} | {'Test':6s} | {'Gap':6s} | {'95% CI':20s}")
print("-" * 85)

for name, m in models.items():
    m.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, m.predict(X_train))
    test_acc = accuracy_score(y_test, m.predict(X_test))
    gap = train_acc - test_acc
    lower, upper = confidence_interval(test_acc, n)
    print(f"{name:30s} | {train_acc:.4f} | {test_acc:.4f} | {gap:.4f} | ({lower:.4f}, {upper:.4f})")

Model                          | Train  | Test   | Gap    | 95% CI              
-------------------------------------------------------------------------------------
Logistic Regression            | 0.4713 | 0.4147 | 0.0566 | (0.3825, 0.4470)
LinearSVC                      | 0.4618 | 0.4047 | 0.0571 | (0.3726, 0.4368)
Naive Bayes                    | 0.4364 | 0.4002 | 0.0362 | (0.3682, 0.4323)
Passive Aggressive             | 0.5809 | 0.1483 | 0.4326 | (0.1250, 0.1715)
Random Forest                  | 0.5917 | 0.1371 | 0.4546 | (0.1146, 0.1596)


In [135]:
from google.colab import drive
drive.mount('/content/drive')

import pickle

with open('/content/drive/MyDrive/model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('/content/drive/MyDrive/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Saved to Drive!")

Mounted at /content/drive
Saved to Drive!
